In [1]:
!git clone https://github.com/guillaumevalette2-hash/mse_gh.git

Cloning into 'mse_gh'...


In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import time
import copy

params = {
    "n_ambiant": 4, "deg_P": 3, "n_terms_poly": 20, "seed": 0, "s_ordre": 3,
        "seeds": {1:  5231, 2: 1769, 3:4186, 42: 3471, 43:739, 44: 539, 11:168, 12:25},
    "weights": {0: 0, 1:1, 2: 0.1, 3: 0.01},
    "n_train": 2000, "levels": 100, "sigma0": 1.5, "sigma_decay": 0.9,
    "n_dict_candidates": 2000, "n_centers_per_level": 200, "n_test": 5000,
    "n_stop": 1e3, "plateau_window": 5, "plateau_tol": 1e-3,
}
params["n_unlabeled"] = 4000 - params["n_train"]
params["lambda_reg"]  = 4 * (params["weights"][3]**(-1)) * (params["n_train"]**(-2))
params["train_center_ratio"] = 0.5 + params["n_train"] / (2 * 4000)

def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree + 1), repeat=d) if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, exp in enumerate(alpha):
                if exp > 0: term *= x[:, j] ** exp
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices) if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_coeff = np.max(np.abs(coeffs))
    if max_coeff > 0:
        normalized_coeffs = coeffs / max_coeff
        def P_normalized(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(normalized_coeffs, indices):
                term = np.ones(x.shape[0])
                for j, exp in enumerate(alpha):
                    if exp > 0:
                        term *= x[:, j] ** exp
                y += c * term
            return y
        return P_normalized
    else:
        return P

P1, indices1, coeffs1 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][1])
P2, indices2, coeffs2 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][2])
P3, indices3, coeffs3 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][3])

P1 = normalize_polynomial(P1, indices1, coeffs1)
P2 = normalize_polynomial(P2, indices2, coeffs2)
P3 = normalize_polynomial(P3, indices3, coeffs3)

def Q(x): return P1(x) * P2(x) * P3(x)

def grad_Q_analytical(X):
    eps = 1e-5; d = X.shape[1]; p1=P1(X); p2=P2(X); p3=P3(X)
    grad = np.zeros_like(X)
    for k in range(d):
        Xp = X.copy(); Xp[:, k] += eps; Xm = X.copy(); Xm[:, k] -= eps
        dp1=(P1(Xp)-P1(Xm))/(2*eps); dp2=(P2(Xp)-P2(Xm))/(2*eps); dp3=(P3(Xp)-P3(Xm))/(2*eps)
        grad[:, k] = dp1*p2*p3 + p1*dp2*p3 + p1*p2*dp3
    return grad

def project_to_Q_zero(X_init, n_steps=150, lr=0.05, tol=1e-4):
    X = X_init.copy()
    for _ in range(n_steps):
        q=Q(X); gq=grad_Q_analytical(X); gq2=2*q[:,None]*gq
        norm=np.linalg.norm(gq2,axis=1,keepdims=True)+1e-12
        X=X-lr*gq2/norm; X=np.clip(X,0.0,1.0)
        if np.abs(Q(X)).max()<tol*0.1: break
    return X, np.abs(Q(X))<tol

def sample_on_Q_zero(n_target, d, tol=1e-4, oversample=4, seed=None):
    rng=np.random.default_rng(seed); collected=[]; n_collected=0
    while n_collected < n_target:
        n_batch=max((n_target-n_collected)*oversample,200)
        X_init=rng.uniform(0,1,(int(n_batch),d)); X_proj,conv=project_to_Q_zero(X_init,tol=tol)
        good=X_proj[conv]
        if len(good)>0: collected.append(good); n_collected+=len(good)
        print(f"  collectés: {n_collected}/{n_target}  (taux convergence batch: {conv.mean():.1%})",end='\r')
    print(); return np.vstack(collected)[:n_target]

print("Génération du cloud sur {Q=0}...")
d = params["n_ambiant"]
X_train     = sample_on_Q_zero(params["n_train"],     d, seed=params["seeds"][42])
X_test      = sample_on_Q_zero(params["n_test"],      d, seed=params["seeds"][43])
X_unlabeled = sample_on_Q_zero(params["n_unlabeled"], d, seed=params["seeds"][44])
print(f"  X_train     : {X_train.shape}  |Q|_max = {np.abs(Q(X_train)).max():.2e}")
print(f"  X_test      : {X_test.shape}   |Q|_max = {np.abs(Q(X_test)).max():.2e}")
print(f"  X_unlabeled : {X_unlabeled.shape}  |Q|_max = {np.abs(Q(X_unlabeled)).max():.2e}")

P_target1, indicest1, coeffst1 =  random_sparse_polynomial(params["n_ambiant"],
                                           7, params["n_terms_poly"], seed= 11)
#def P_target1_normalized(X):
 #   return P_target1(X) / ((P_target1(_X0)**2 + 1)**0.5)
#P_target2, _, _ = random_sparse_polynomial(params["n_ambiant"],
#                                           7, params["n_terms_poly"], seed= 12)
Ptarget1 = normalize_polynomial(P_target1, indicest1, coeffst1)

def target_function(X):
    return  5  * np.sin (P_target1(3*X))
#X[:,3]**2+3+20*np.maximum(np.sin(X[:,0]*X[:,1]**3-8*X[:,2]**3+7*X[:,3]**9),0)
y_train = target_function(X_train); y_test = target_function(X_test)
X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)

# ── PATCH 1 : dérivées conditionnelles selon weights ─────────────────────────
def gaussian_phi_grad_hess(X, centers, sigma, weights=None):
    d=X.shape[1]; s2=sigma**2
    diff=X[:,None,:]-centers[None,:,:]; sq=np.sum(diff**2,axis=2); phi=np.exp(-sq/(2*s2))
    need_grad = weights is None or weights.get(1,0.)!=0 or weights.get(2,0.)!=0
    grad = -(diff/s2)*phi[:,:,None] if need_grad else None
    need_hess = weights is None or weights.get(2,0.)!=0
    if need_hess:
        s4=sigma**4
        hess=(np.einsum('nmi,nmj->nmij',diff,diff)/s4-np.eye(d)[None,None,:,:]/s2)*phi[:,:,None,None]
    else: hess=None
    return phi, grad, hess

def gaussian_all_derivatives(X, centers, sigma, weights=None):
    d=X.shape[1]; s2=sigma**2
    diff=X[:,None,:]-centers[None,:,:]; sq=np.sum(diff**2,axis=2); phi=np.exp(-sq/(2*s2))
    grad=-(diff/s2)*phi[:,:,None]
    need_hess = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    if need_hess:
        s4=sigma**4
        hess=(np.einsum('nmi,nmj->nmij',diff,diff)/s4-np.eye(d)[None,None,:,:]/s2)*phi[:,:,None,None]
    else: hess=None
    need_third = weights is None or weights.get(3,0.)!=0
    if need_third:
        s4=sigma**4; s6=sigma**6
        cubic=np.einsum('nmi,nmj,nmk->nmijk',diff,diff,diff)/s6
        Id=np.eye(d)
        sym=(np.einsum('ij,nmk->nmijk',Id,diff)+np.einsum('ik,nmj->nmijk',Id,diff)+np.einsum('jk,nmi->nmijk',Id,diff))/s4
        third=(-cubic+sym)*phi[:,:,None,None,None]
    else: third=None
    return phi, grad, hess, third

def gaussian_features(X, centers, sigma):
    diff=X[:,None,:]-centers[None,:,:]; sq=np.sum(diff**2,axis=2); return np.exp(-sq/(2*sigma**2))

def sobolev_norms_sq_partial(phi, grad, hess, weights, n):
    norms=np.zeros(phi.shape[1])
    w0=weights.get(0,0.);
    if w0: norms+=w0*np.einsum('xj,xj->j',phi,phi)/n
    w1=weights.get(1,0.)
    if w1 and grad is not None: norms+=w1*np.einsum('xjk,xjk->j',grad,grad)/n
    w2=weights.get(2,0.)
    if w2 and hess is not None: norms+=w2*np.einsum('xjkl,xjkl->j',hess,hess)/n
    return norms

def sobolev_norms_sq(phi, grad, hess, third, weights, n):
    norms=sobolev_norms_sq_partial(phi,grad,hess,weights,n)
    w3=weights.get(3,0.)
    if w3 and third is not None: norms+=w3*np.einsum('xjklm,xjklm->j',third,third)/n
    return norms

def build_G(phi, grad, hess, third, weights, n):
    G=np.zeros((phi.shape[1],phi.shape[1]))
    w0=weights.get(0,0.)
    if w0: G+=w0*(phi.T@phi)/n
    w1=weights.get(1,0.)
    # ── PATCH 2 : optimize='optimal' sur les einsum coûteux ──────────────────
    if w1 and grad  is not None: G+=w1*np.einsum('xik,xjk->ij',    grad, grad, optimize='optimal')/n
    w2=weights.get(2,0.)
    if w2 and hess  is not None: G+=w2*np.einsum('xikl,xjkl->ij',  hess, hess, optimize='optimal')/n
    w3=weights.get(3,0.)
    if w3 and third is not None: G+=w3*np.einsum('xiklm,xjklm->ij',third,third,optimize='optimal')/n
    return G

def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr=min(int(round(n_candidates*train_ratio)),X_train.shape[0])
    n_ul=min(n_candidates-n_tr,X_unlabeled.shape[0]); parts=[]
    if n_tr>0: parts.append(X_train[rng.choice(X_train.shape[0],n_tr,replace=False)])
    if n_ul>0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0],n_ul,replace=False)])
    return np.vstack(parts)

losses=[]; times=[]; pred_train_accum=np.zeros(len(X_train)); pred_test_accum=np.zeros(len(X_test))
fi_derivs_all=None; history=[]; stop_reason=None; final_level=None

for level in range(params["levels"]):
    t0=time.time()
    sigma=params["sigma0"]*params["sigma_decay"]**level
    lambda_reg=params["lambda_reg"]*(sigma/params["sigma0"])**2
    lambda_reg=max(lambda_reg,1e-7)
    rng=np.random.default_rng(seed=level)

    candidates=sample_candidates(X_train,X_unlabeled,params["n_dict_candidates"],params["train_center_ratio"],rng)

    phi_c,grad_c,hess_c=gaussian_phi_grad_hess(X_all,candidates,sigma,params["weights"])
    A_cand=gaussian_features(X_train,candidates,sigma)
    residual=y_train-pred_train_accum
    corr=(A_cand.T@residual)/len(X_train)

    norms_sq=sobolev_norms_sq_partial(phi_c,grad_c,hess_c,params["weights"],N_all)
    norms_sq=np.maximum(norms_sq,1e-12); scores=corr**2/norms_sq

    k=min(params["n_centers_per_level"],len(candidates)); top_k=np.argsort(scores)[-k:]
    centers=candidates[top_k]; A=A_cand[:,top_k]; Atest=gaussian_features(X_test,centers,sigma)
    phi,grad,hess,third=gaussian_all_derivatives(X_all,centers,sigma,params["weights"])

    n_G=int(min(N_all,max(500,int(-params["n_train"]+8*sigma**(-3)))));n_G=min(n_G,2000)
    idx_G=rng.choice(N_all,size=n_G,replace=False)
    G=build_G(phi[idx_G],
              grad[idx_G]  if grad  is not None else None,
              hess[idx_G]  if hess  is not None else None,
              third[idx_G] if third is not None else None,
              params["weights"],n_G)
    n=A.shape[0]; M=(A.T@A)/n+lambda_reg*G; rhs=(A.T@residual)/n

    print(f"\n=== LEVEL {level} | σ={sigma:.4f} | λ={lambda_reg:.2e} | dict={len(candidates)} → k={k} | n_G={n_G} | score max={scores[top_k[-1]]:.3e} min={scores[top_k[0]]:.3e} ===")

    if fi_derivs_all is not None:
        b=np.zeros(k)
        w0=params["weights"].get(0,0.)
        if w0: b+=w0*(fi_derivs_all['phi']@phi)/N_all
        w1=params["weights"].get(1,0.)
        if w1 and grad  is not None: b+=w1*np.einsum('xk,xjk->j',  fi_derivs_all['grad'], grad, optimize='optimal')/N_all
        w2=params["weights"].get(2,0.)
        if w2 and hess  is not None: b+=w2*np.einsum('xkl,xjkl->j',fi_derivs_all['hess'], hess, optimize='optimal')/N_all
        w3=params["weights"].get(3,0.)
        if w3 and third is not None: b+=w3*np.einsum('xklm,xjklm->j',fi_derivs_all['third'],third,optimize='optimal')/N_all
        rhs=rhs-lambda_reg*b

    eigvals,eigvecs=np.linalg.eigh(M); thresh=max(lambda_reg,1e-7); mask=eigvals>thresh
    V=eigvecs[:,mask]; S=eigvals[mask]; coeffs=V@((V.T@rhs)/S)

    loss_data=np.mean((residual-A@coeffs)**2); loss_reg=lambda_reg*coeffs@G@coeffs
    loss_total=loss_data+loss_reg; t1=time.time(); times.append(t1-t0)

    print(f"  rang={mask.sum()}/{k}  |coeff|_max={np.max(np.abs(coeffs)):.4f}  temps={times[-1]:.1f}s")
    print(f"  loss={loss_total:.6f}  (data={loss_data:.6f}  reg={loss_reg:.6f})")

    if len(history)>0:
        prev_loss=history[-1]['loss_total']
        if loss_total>prev_loss+params["n_stop"]*lambda_reg:
            best_idx=len(history)-1
            for i in range(len(history)-2,-1,-1):
                if history[i]['loss_total']<=history[i+1]['loss_total']: best_idx=i
                else: break
            print(f"  ⚠ ARRÊT : explosion détectée")
            print(f"  → séquence de hausses remonte au niveau {history[best_idx]['level']+1}")
            print(f"  → on garde le niveau {history[best_idx]['level']+1} comme résultat final")
            stop_reason="explosion"; final_level=best_idx; break

    pred_train_accum_new=pred_train_accum+A@coeffs; pred_test_accum_new=pred_test_accum+Atest@coeffs
    delta_phi=phi@coeffs
    delta_grad =np.einsum('xjk,j->xk',   grad, coeffs) if grad  is not None else None
    delta_hess =np.einsum('xjkl,j->xkl', hess, coeffs) if hess  is not None else None
    delta_third=np.einsum('xjklm,j->xklm',third,coeffs) if third is not None else None

    def _add(a,b): return a+b if (a is not None and b is not None) else None
    if fi_derivs_all is None:
        fi_derivs_all_new={'phi':delta_phi,'grad':delta_grad,'hess':delta_hess,'third':delta_third}
    else:
        fi_derivs_all_new={'phi':fi_derivs_all['phi']+delta_phi,
                           'grad': _add(fi_derivs_all['grad'], delta_grad),
                           'hess': _add(fi_derivs_all['hess'], delta_hess),
                           'third':_add(fi_derivs_all['third'],delta_third)}

    mse=mean_squared_error(y_test,pred_test_accum_new); losses.append(mse)
    print(f"  MSE level {level+1} = {mse:.6f}")
    pred_train_accum=pred_train_accum_new; pred_test_accum=pred_test_accum_new; fi_derivs_all=fi_derivs_all_new
    history.append({'level':level,'pred_train':pred_train_accum.copy(),'pred_test':pred_test_accum.copy(),
                    'fi_derivs':copy.deepcopy(fi_derivs_all),'mse':mse,'loss_total':loss_total})

    w=params["plateau_window"]
    if len(history)>=w:
        recent_losses=[h['loss_total'] for h in history[-w:]]
        improvement=recent_losses[0]-recent_losses[-1]
        if improvement<params["plateau_tol"]:
            print(f"  ⚠ ARRÊT : plateau détecté — baisse de loss sur {w} niveaux = {improvement:.6f} (< tol={params['plateau_tol']})")
            print(f"  → on garde le niveau {history[-w]['level']+1} (premier de la fenêtre de plateau) comme résultat final")
            stop_reason="plateau"; final_level=len(history)-2; break

if stop_reason is None: final_level=len(history)-1; stop_reason="max_levels"
final_state=history[final_level]; mse_final=final_state['mse']
print("\n"+"="*80)
print(f"ARRÊT : raison = {stop_reason}")
print(f"Niveau final retenu : {final_state['level']+1} / {params['levels']} effectués")
print(f"MSE final (sur le niveau retenu) : {mse_final:.6f}")
print("="*80)

from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

# =====================================================
# COMPARAISONS
# =====================================================
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

# ── RBF avec GridSearchCV (uniquement) ────────────────────────────────────
print("\nCalibration RBF (GridSearchCV)...")
gamma_grid = np.logspace(-3, 2, 20)  # 20 valeurs de gamma entre 1e-3 et 1e2
param_grid = {
    'alpha': [1e-8, 1e-5, 1e-3],  # 3 valeurs de alpha
    'gamma': gamma_grid,
    'kernel': ['rbf']
}
kr_cv = GridSearchCV(
    KernelRidge(),
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1  # Utilise tous les cœurs disponibles
)
kr_cv.fit(X_train, y_train)

# Récupère le meilleur modèle pour la comparaison finale
best_gamma = kr_cv.best_params_['gamma']
best_alpha = kr_cv.best_params_['alpha']
kr_best = KernelRidge(kernel='rbf', gamma=best_gamma, alpha=best_alpha)
kr_best.fit(X_train, y_train)
best_err_rbf = mean_squared_error(y_test, kr_best.predict(X_test))

# Affichage des Top 3 RBF
results = kr_cv.cv_results_
order = np.argsort(results['rank_test_score'])
print("\nTop 3 RBF :")
for i in range(3):
    idx = order[i]
    gamma = results['param_gamma'][idx]
    alpha = results['param_alpha'][idx]
    kr = KernelRidge(kernel='rbf', gamma=float(gamma), alpha=float(alpha))
    kr.fit(X_train, y_train)
    mse_test = mean_squared_error(y_test, kr.predict(X_test))
    print(f"#{i+1}  gamma={gamma:.4f}  alpha={alpha:.2e}  CV={-results['mean_test_score'][idx]:.6f}  TEST={mse_test:.6f}")

# ── Polynomial Ridge ─────────────────────────────────────────────────────
poly = PolynomialFeatures(degree=min(params["deg_P"], 8), include_bias=False)
ridge_poly = Ridge(alpha=1e-8)
ridge_poly.fit(poly.fit_transform(X_train), y_train)
err_poly = mean_squared_error(y_test, ridge_poly.predict(poly.transform(X_test)))

print("\n"+"="*80); print("HYPERPARAMÈTRES"); print("="*80)
print(f"kernel               = Gaussien vectorisé")
print(f"n_dict_candidates    = {params['n_dict_candidates']}")
print(f"n_centers_per_level  = {params['n_centers_per_level']}")
print(f"n_stop               = {params['n_stop']}  (seuil d'explosion = n_stop·λ)")
print(f"plateau_window       = {params['plateau_window']}  plateau_tol={params['plateau_tol']}")
print(f"train_center_ratio   = {params['train_center_ratio']}")
print(f"levels (max)         = {params['levels']}")
print(f"sigma0               = {params['sigma0']}  decay={params['sigma_decay']}")
print(f"lambda_reg (lvl 0)   = {params['lambda_reg']}  (adaptatif × (σ/σ0)²)")
print(f"weights              = {params['weights']}")
print(f"n_train={params['n_train']}  n_unlabeled={params['n_unlabeled']}  n_test={params['n_test']}")
print(f"temps total boucle   = {sum(times):.1f}s  (moy {np.mean(times):.1f}s/niveau)")
print("\n"+"="*80); print("RÉSULTATS"); print("="*80)
print(f"Polynomial Ridge     : {err_poly:.6f}")
print(f"RBF (best γ={best_gamma:.4f}, α={best_alpha:.2e}) : {best_err_rbf:.6f}")
print(f"Sobolev MS Gaussien  : {mse_final:.6f}  (niveau {final_state['level']+1}, arrêt={stop_reason})")
print("\nDécroissance par niveau (effectués) :")
for i,loss in enumerate(losses):
    marker="  ← retenu" if i==final_level else ""
    print(f"  Niveau {i+1:2d} : {loss:.6f}  ({times[i]:.1f}s){marker}")


Génération du cloud sur {Q=0}...
  collectés: 3923/2000  (taux convergence batch: 49.0%)
  collectés: 9772/5000  (taux convergence batch: 48.9%)
  collectés: 3934/2000  (taux convergence batch: 49.2%)
  X_train     : (2000, 4)  |Q|_max = 9.96e-05
  X_test      : (5000, 4)   |Q|_max = 9.99e-05
  X_unlabeled : (2000, 4)  |Q|_max = 9.98e-05

=== LEVEL 0 | σ=1.5000 | λ=1.00e-04 | dict=2000 → k=400 | n_G=500 | score max=1.047e+00 min=9.465e-01 ===
  rang=5/400  |coeff|_max=0.5102  temps=10.2s
  loss=8.105328  (data=8.104235  reg=0.001092)
  MSE level 1 = 8.460575

=== LEVEL 1 | σ=1.2000 | λ=6.40e-05 | dict=2000 → k=400 | n_G=500 | score max=2.336e-05 min=1.712e-05 ===
  rang=5/400  |coeff|_max=0.0087  temps=9.5s
  loss=8.104187  (data=8.104187  reg=0.000000)
  MSE level 2 = 8.462467

=== LEVEL 2 | σ=0.9600 | λ=4.10e-05 | dict=2000 → k=400 | n_G=500 | score max=2.929e-05 min=1.644e-05 ===
  rang=9/400  |coeff|_max=21.5479  temps=9.9s
  loss=7.916406  (data=7.913798  reg=0.002609)
  MSE level